# Lab 3 - Image Captioning (Task 3.2)

Image captioning network on Flickr8k. CNN encoder (ResNet50) + LSTM decoder, with a switchable fusion step (concat / add / mult / diff) so we can compare them across the group.

The notebook follows the steps from the assignment - each section below has its own code block:

| Section | Code |
|---|---|
| 1. Image Feature Extraction | `EncoderCNN` - frozen ResNet50 + projection |
| 2. Sequence Model for Language Processing | `TextEncoder` - embedding + LSTM, returns hidden states |
| 3. Combining Image and Text Data | `Combine` - image projection + fusion (concat/add/mult/diff) + output FC |
| 4. Training the Network | training loop |
| 5. Generating Captions | greedy decode + sample plot + wandb table |
| 6. Evaluation | BLEU-1 / BLEU-4 |

Setup / config / vocab / dataset cells come first, before the numbered sections.

## Setup

In [1]:
!pip install -q nltk wandb
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

### Config

Each group member runs one of: `concat`, `add`, `mult`, `diff`.

In [ ]:
fusions = ['concat', 'add', 'mult', 'diff']

dataset_dir = './archive'
image_dir = dataset_dir + '/Images'
caption_file = dataset_dir + '/captions.txt'

embed_dim = 256
hidden_dim = 256
encoder_dim = 2048  # resnet50 output

batch_size = 32
num_epochs = 15
lr = 1e-3
max_len = 40
freq_thresh = 5

seed = 42

In [3]:
import os
import re
import random
import numpy as np
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import matplotlib.pyplot as plt
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import wandb

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

device: cuda
NVIDIA GeForce RTX 2080 Ti


In [ ]:
wandb.init(
    project='D7047E-Lab3',
    name='captioning-all-fusions',
    config={
        'fusions': fusions,
        'embed_dim': embed_dim,
        'hidden_dim': hidden_dim,
        'encoder_dim': encoder_dim,
        'lr': lr,
        'batch_size': batch_size,
        'epochs': num_epochs,
        'max_len': max_len,
        'freq_thresh': freq_thresh,
        'encoder': 'resnet50',
        'seed': seed,
    },
)

### Vocabulary

Build word2idx from captions.txt. Words seen fewer than `freq_thresh` times collapse to UNK.
Specials: PAD=0, START=1, END=2, UNK=3.

In [5]:
PAD, START, END, UNK = 0, 1, 2, 3

def tokenize(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

class Vocabulary:
    def __init__(self, freq_threshold):
        self.freq_threshold = freq_threshold
        self.word2idx = {'<PAD>': PAD, '<START>': START, '<END>': END, '<UNK>': UNK}
        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def build(self, captions):
        counter = Counter()
        for c in captions:
            counter.update(tokenize(c))
        i = len(self.word2idx)
        for word, cnt in counter.items():
            if cnt >= self.freq_threshold:
                self.word2idx[word] = i
                self.idx2word[i] = word
                i += 1
        print('vocab size:', len(self.word2idx))

    def encode(self, caption):
        ids = [self.word2idx.get(w, UNK) for w in tokenize(caption)]
        return [START] + ids + [END]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == END:
                break
            if i in (PAD, START):
                continue
            out.append(self.idx2word.get(i, '<UNK>'))
        return ' '.join(out)

    def __len__(self):
        return len(self.word2idx)

In [6]:
# load captions.txt (header + 'image_name,caption' per line)
all_captions = []
all_data = []

with open(caption_file, 'r') as f:
    next(f)  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(',', 1)
        if len(parts) != 2:
            continue
        img, cap = parts[0].strip(), parts[1].strip()
        all_data.append((img, cap))
        all_captions.append(cap)

print(len(all_data), 'pairs,', len(set(d[0] for d in all_data)), 'unique images')
print('example:', all_data[0])

vocab = Vocabulary(freq_thresh)
vocab.build(all_captions)
wandb.config.update({'vocab_size': len(vocab)})

40455 pairs, 8091 unique images
example: ('1000268201_693b08cb0e.jpg', 'A child in a pink dress is climbing up a set of stairs in an entry way .')
vocab size: 2988


### Dataset / DataLoader

Splitting by image (not by pair) so the same image doesn't end up in both train and test.

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FlickrDataset(Dataset):
    def __init__(self, data, image_dir, vocab, transform, max_len):
        self.data = data
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        name, caption = self.data[idx]
        img = Image.open(os.path.join(self.image_dir, name)).convert('RGB')
        img = self.transform(img)

        ids = self.vocab.encode(caption)
        if len(ids) < self.max_len:
            ids = ids + [PAD] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len - 1] + [END]
        return img, torch.tensor(ids, dtype=torch.long)

unique_imgs = list({d[0] for d in all_data})
random.shuffle(unique_imgs)
cut = int(0.9 * len(unique_imgs))
train_set = set(unique_imgs[:cut])
test_set = set(unique_imgs[cut:])

train_data = [(n, c) for n, c in all_data if n in train_set]
test_data = [(n, c) for n, c in all_data if n in test_set]
print('train:', len(train_data), ' test:', len(test_data))

train_ds = FlickrDataset(train_data, image_dir, vocab, transform, max_len)
test_ds = FlickrDataset(test_data, image_dir, vocab, transform, max_len)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

train: 36405  test: 4050


## 1. Image Feature Extraction

Pretrained ResNet50 with the classification head removed. The 2048-dim avgpool feature is projected down to `embed_dim` by a trainable FC + BN. Backbone weights are frozen, so only the projection layer learns.

In [8]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_dim, encoder_dim=2048):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # drop the final fc, keep up to avgpool
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        for p in self.resnet.parameters():
            p.requires_grad = False
        self.fc = nn.Linear(encoder_dim, embed_dim)
        self.bn = nn.BatchNorm1d(embed_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, images):
        with torch.no_grad():
            x = self.resnet(images)
        x = x.view(x.size(0), -1)
        x = self.dropout(self.bn(self.fc(x)))
        return x

encoder = EncoderCNN(embed_dim, encoder_dim).to(device)
print('trainable encoder params:', sum(p.numel() for p in encoder.parameters() if p.requires_grad))

trainable encoder params: 525056


## 2. Sequence Model for Language Processing

Word embedding -> single-layer LSTM. Takes the caption tokens (teacher forcing during training) and returns the per-step hidden states, which we'll combine with the image vector in the next section.

In [9]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, batch_first=True)

    def forward(self, captions):
        e = self.embedding(captions)         # (B, T, embed_dim)
        out, _ = self.lstm(e)                # (B, T, hidden_dim)
        return out

text_model = TextEncoder(len(vocab), embed_dim, hidden_dim).to(device)
print('text model params:', sum(p.numel() for p in text_model.parameters()))

text model params: 1291264


## 3. Combining Image and Text Data

Merge architecture (Tanti & Gatt) - image embedding and the LSTM hidden states are kept independent and combined per timestep, then a single FC predicts the next word.

Fusion options:
- `concat`: stack along last dim, then linear over `embed_dim + hidden_dim`
- `add` / `mult` / `diff`: project the LSTM output to `embed_dim` first, then elementwise op, then linear over `embed_dim`

In [ ]:
class Combine(nn.Module):
    def __init__(self, embed_dim, hidden_dim, encoder_dim, vocab_size, fusion='concat'):
        super().__init__()
        self.fusion = fusion

        # image side projection (ResNet feat -> embed_dim)
        self.img_fc = nn.Linear(encoder_dim, embed_dim)
        self.img_bn = nn.BatchNorm1d(embed_dim)

        if fusion == 'concat':
            self.out_fc = nn.Linear(embed_dim + hidden_dim, vocab_size)
        else:
            # need same dim for elementwise ops
            self.txt_proj = nn.Linear(hidden_dim, embed_dim)
            self.out_fc = nn.Linear(embed_dim, vocab_size)

        self.dropout = nn.Dropout(0.3)

    def fuse(self, img, txt):
        if self.fusion == 'concat':
            return torch.cat([img, txt], dim=-1)
        txt = self.txt_proj(txt)
        if self.fusion == 'add':
            return img + txt
        if self.fusion == 'mult':
            return img * txt
        if self.fusion == 'diff':
            return img - txt
        raise ValueError(self.fusion)

    def forward(self, image_features, lstm_out):
        img = torch.relu(self.img_bn(self.img_fc(image_features)))   # (B, embed_dim)
        img = img.unsqueeze(1).expand(-1, lstm_out.size(1), -1)      # broadcast over time
        fused = self.dropout(self.fuse(img, lstm_out))
        return self.out_fc(fused)                                    # (B, T, V)

# class is instantiated per-fusion in Section 4

## 4. Training the Network

Train one model per fusion. For each fusion we build a fresh `EncoderCNN` + `TextEncoder` + `Combine` and train with cross-entropy (`ignore_index=PAD`) under teacher forcing. Models are kept in a `models[fusion] = (encoder, text_model, combine)` dict and the final epoch loss is recorded in `metrics[fusion]['train_loss']` for the comparison at the end.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

models = {}
metrics = {}

for f in fusions:
    print(f'\n=== training fusion: {f} ===')
    enc = EncoderCNN(embed_dim, encoder_dim).to(device)
    txt = TextEncoder(len(vocab), embed_dim, hidden_dim).to(device)
    cmb = Combine(embed_dim, hidden_dim, embed_dim, len(vocab), fusion=f).to(device)

    prms = (list(enc.fc.parameters()) + list(enc.bn.parameters())
            + list(txt.parameters()) + list(cmb.parameters()))
    opt = optim.Adam(prms, lr=lr)

    last_loss = 0.0
    for epoch in range(num_epochs):
        enc.train(); txt.train(); cmb.train()
        running, n = 0.0, 0
        for images, captions in train_loader:
            images = images.to(device); captions = captions.to(device)
            x, y = captions[:, :-1], captions[:, 1:]
            feats = enc(images)
            lstm_out = txt(x)
            logits = cmb(feats, lstm_out)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(prms, max_norm=5.0)
            opt.step()
            running += loss.item(); n += 1
        last_loss = running / n
        wandb.log({f'{f}/train_loss': last_loss, f'{f}/epoch': epoch + 1})
        print(f'  [{f}] epoch {epoch+1}/{num_epochs}  loss={last_loss:.4f}')

    models[f] = (enc, txt, cmb)
    metrics[f] = {'train_loss': last_loss}

print('\nall fusions trained.')

## 5. Generating Captions

Greedy decoding for each trained fusion. Same set of test images is captioned by all four models so the qualitative differences are easy to compare side-by-side.

In [ ]:
@torch.no_grad()
def generate_caption(image, enc, txt, cmb, max_len=25):
    enc.eval(); txt.eval(); cmb.eval()
    image = image.unsqueeze(0).to(device)
    feats = enc(image)
    seq = [START]
    for _ in range(max_len):
        inp = torch.tensor([seq], dtype=torch.long, device=device)
        lstm_out = txt(inp)
        logits = cmb(feats, lstm_out)
        nxt = logits[0, -1].argmax().item()
        seq.append(nxt)
        if nxt == END:
            break
    return vocab.decode(seq)

@torch.no_grad()
def caption_from_path(path, enc, txt, cmb):
    img = Image.open(path).convert('RGB')
    return generate_caption(transform(img), enc, txt, cmb)

In [ ]:
test_imgs = list({n for n, _ in test_data})
random.shuffle(test_imgs)
samples = test_imgs[:5]

for f, (enc, txt, cmb) in models.items():
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for i, name in enumerate(samples):
        path = os.path.join(image_dir, name)
        img = Image.open(path).convert('RGB')
        pred = caption_from_path(path, enc, txt, cmb)
        refs = [c for n, c in all_data if n == name]
        ref = refs[0] if refs else ''
        axes[i].imshow(img)
        axes[i].set_title(f'pred: {pred}\nref:  {ref[:60]}', fontsize=8)
        axes[i].axis('off')
    plt.suptitle('fusion = ' + f)
    plt.tight_layout()
    plt.savefig(f'sample_captions_{f}.png', dpi=150, bbox_inches='tight')
    plt.show()
    wandb.log({f'samples_{f}': wandb.Image(f'sample_captions_{f}.png')})

In [ ]:
for f, (enc, txt, cmb) in models.items():
    table = wandb.Table(columns=['image', 'generated', 'reference', 'name'])
    for name in samples:
        path = os.path.join(image_dir, name)
        pred = caption_from_path(path, enc, txt, cmb)
        refs = [c for n, c in all_data if n == name]
        ref = refs[0] if refs else ''
        table.add_data(wandb.Image(path, caption=pred), pred, ref, name)
    wandb.log({f'caption_samples_{f}': table})

## 6. Evaluation

Corpus BLEU-1 / BLEU-4 on the test set for each fusion, against all available reference captions per image. Scores are written into the `metrics` dict for the comparison in Section 7.

In [ ]:
ref_dict = {}
for name, cap in test_data:
    ref_dict.setdefault(name, []).append(tokenize(cap))

smooth = SmoothingFunction().method1

for f, (enc, txt, cmb) in models.items():
    references, hypotheses = [], []
    for name in ref_dict:
        path = os.path.join(image_dir, name)
        if not os.path.exists(path):
            continue
        pred = caption_from_path(path, enc, txt, cmb).split()
        references.append(ref_dict[name])
        hypotheses.append(pred)

    b1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smooth)
    b4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    metrics[f]['bleu1'] = b1
    metrics[f]['bleu4'] = b4
    print(f'{f:>6}  BLEU-1: {b1:.4f}  BLEU-4: {b4:.4f}')
    wandb.log({f'{f}/bleu1': b1, f'{f}/bleu4': b4})

### Save + finish wandb

In [ ]:
ckpt = {
    'models': {f: {'encoder_state_dict': enc.state_dict(),
                   'text_model_state_dict': txt.state_dict(),
                   'combine_state_dict': cmb.state_dict()}
               for f, (enc, txt, cmb) in models.items()},
    'metrics': metrics,
    'vocab': vocab,
    'config': {
        'embed_dim': embed_dim,
        'hidden_dim': hidden_dim,
        'vocab_size': len(vocab),
        'fusions': fusions,
    },
}
torch.save(ckpt, 'captioning_all_fusions.pth')
print('saved captioning_all_fusions.pth')

wandb.finish()

## 7. Fusion Comparison

Sections 4 and 6 already trained and evaluated all four fusions. This section pulls the numbers from `metrics` into a single table + bar chart for side-by-side comparison.

In [ ]:
import pandas as pd

df = pd.DataFrame([{'fusion': f, **m} for f, m in metrics.items()])
df = df.round(4)
print(df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.plot.bar(x='fusion', y='train_loss', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('Final Train Loss'); axes[0].set_ylabel('loss')
df.plot.bar(x='fusion', y='bleu1', ax=axes[1], legend=False, color='seagreen')
axes[1].set_title('BLEU-1'); axes[1].set_ylabel('score')
df.plot.bar(x='fusion', y='bleu4', ax=axes[2], legend=False, color='coral')
axes[2].set_title('BLEU-4'); axes[2].set_ylabel('score')
for ax in axes:
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('fusion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

wandb.log({'fusion_comparison_plot': wandb.Image('fusion_comparison.png')})

results_table = wandb.Table(columns=['fusion', 'train_loss', 'bleu1', 'bleu4'])
for _, row in df.iterrows():
    results_table.add_data(row['fusion'], row['train_loss'], row['bleu1'], row['bleu4'])
wandb.log({'fusion_comparison_table': results_table})

### Discussion

**What each fusion does**

- **concat**: stacks the image vector and the LSTM hidden state along the feature axis, then a single FC over `embed_dim + hidden_dim` learns the interaction. This is the most flexible variant - the network itself decides how the two modalities mix, at the cost of a wider output layer (more parameters).
- **add**: projects the LSTM output to `embed_dim` and elementwise-adds the image. Cheap and stable, but every output dimension is forced to carry a mix of both inputs at the same scale - it can't dedicate dimensions to one modality.
- **mult**: elementwise product of image and projected text. Behaves like a soft gate: a near-zero on either side zeros the output. This is a strong inductive bias (good when the streams should mutually agree) but training is harder because gradients shrink whenever either side is small.
- **diff**: image minus projected text. Encodes how the image differs from what language has already produced. Less standard for captioning - subtraction throws away symmetric information and is more typical in metric/contrastive setups.

**How to read the table**

- *BLEU-1* measures unigram overlap. It mostly tells you whether the model picks the right object/attribute words (e.g. "dog", "running", "grass"). A fusion with low BLEU-1 is failing at content selection.
- *BLEU-4* measures 4-gram overlap. It rewards fluent, well-structured sentences. Models can have similar BLEU-1 but different BLEU-4 if one produces grammatically tighter captions.
- *Train loss* shows fitting capacity, not generalization. A fusion with the lowest train loss isn't automatically the best - if BLEU lags it has overfit.

**What we typically expect, and why**

- `concat` and `add` usually lead on BLEU because they preserve information from both streams without forcing them to interact in a restrictive way.
- `mult` is competitive when both encoders are well-calibrated, but often converges slower and can underperform if either stream produces small magnitudes.
- `diff` is usually the weakest. The image vector is identical at every timestep (no attention), so once the LSTM hidden state grows, `img - txt` is dominated by `-txt` and the visual signal washes out.

**How to interpret your specific numbers above**

- If `concat` wins on both BLEUs - expected, it has the most expressive output layer.
- If `add` matches `concat` - the extra capacity of concat wasn't needed; useful evidence for a more parameter-efficient model.
- If `mult` wins BLEU-1 but loses BLEU-4 - the gating helps with correct word choice but hurts sentence structure (gradient issues during longer decodes).
- If `diff` is close to the others - probably because the image projection is small enough that the LSTM doesn't drown it out; with longer training this gap usually grows.